# Phase 10.8 — LTR Inference & Baseline Comparison

Offline LTR inference and metric comparison vs the Phase 10.4 baseline.

**Production:** `retrieve_ranked()` still uses the 10.4 baseline only.

**Limitation:** 10-query lexical benchmark / small LTR artifact — demo only, not generalization evidence.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.ltr import (
    LTRBaselineBenchmarkEvaluator,
    LTRTrainingConfig,
    load_validated_ltr_artifact,
    rank_candidates_with_ltr,
    train_ltr_model,
)
from productiq.ranking.evaluation import RankingEvaluationConfig
from productiq.ranking.ltr.compatibility import validate_ltr_artifact_for_pipeline
from productiq.ranking.dataset_builder import RankingDatasetQueryGroup, build_ranking_dataset_from_query_groups
from productiq.ranking.feature_schema import RankingFeatures
from productiq.retrieval.evaluation.dataset import load_lexical_retrieval_benchmark, default_benchmark_path

In [ ]:
def feature_row(product_id: str) -> RankingFeatures:
    return RankingFeatures(
        product_id=product_id,
        retrieval={"bm25_score": 1.0, "rrf_score": 0.5, "retrieved_by_bm25": True},
        matching={"matched_attribute_count": 1.0},
        catalog={},
    )

dataset = build_ranking_dataset_from_query_groups(
    (
        RankingDatasetQueryGroup(
            query_id="NB_Q1",
            features=(feature_row("P1"), feature_row("P2")),
            relevance_label_by_product_id={"P1": 1, "P2": 0},
        ),
    ),
    query_set_version="notebook-10.8",
    judgment_source_version="fixture",
    require_labels=True,
)
artifact_dir = ROOT / "resources" / "models" / "ranking_ltr_reference_v10_7_0"
if not (artifact_dir / "model.lgb").is_file():
    train_ltr_model(dataset, LTRTrainingConfig(num_boost_round=10), artifact_directory=artifact_dir)
artifact = load_validated_ltr_artifact(artifact_dir)
validate_ltr_artifact_for_pipeline(artifact)
artifact.metadata.feature_spec.feature_names[:3]

In [ ]:
benchmark = load_lexical_retrieval_benchmark(default_benchmark_path(ROOT))
eval_config = RankingEvaluationConfig()
print(benchmark.benchmark_name, benchmark.benchmark_version, len(benchmark.queries), "queries")

## Comparison from explicit ranked lists (same pool)

Production pipeline comparison requires a configured `ProductionRetrievalPipeline` with catalog + index artifacts. Below demonstrates the offline evaluator contract with list inputs aligned to the benchmark.

In [ ]:
evaluator = LTRBaselineBenchmarkEvaluator(eval_config)
demo = evaluator.evaluate_from_lists(
    benchmark,
    baseline_by_query_id={q.query_id: q.relevant_product_ids for q in benchmark.queries},
    ltr_by_query_id={q.query_id: tuple(reversed(q.relevant_product_ids)) for q in benchmark.queries},
    pool_by_query_id={q.query_id: q.relevant_product_ids for q in benchmark.queries},
    artifact=artifact,
)
print("evaluation_version", demo.evaluation_version)
print("lineage", demo.lineage.model_dump())
print("MRR baseline", demo.baseline.mrr, "ltr", demo.ltr.mrr, "delta", demo.comparison.mrr.delta)

**Limitations:** Judgments are incomplete; pool construction here is illustrative. Do not treat deltas as optimization guidance.